## When sessions don't add up

In [ ]:
import os
from pathlib import Path
import pandas as pd

from dotenv import find_dotenv, load_dotenv
from google.cloud import bigquery

load_dotenv(find_dotenv(usecwd=True))
project_id = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not project_id:
    raise RuntimeError("Set GOOGLE_CLOUD_PROJECT in a .env file (see ../.env.example).")

# Queries run in your own project: billed there, or counted against the free sandbox quota.
client = bigquery.Client(project=project_id)

# Paths are relative to this notebook's folder.
SQL_DIR = Path("sql")
RAW_DIR = Path("csv")


def run_query(name):
    """Run sql/<name>.sql and return the result as a DataFrame."""
    sql = (SQL_DIR / f"{name}.sql").read_text(encoding="utf-8")
    # A dry run is free and shows the cost before the real run.
    estimate = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True))
    print(f"{name}: {estimate.total_bytes_processed / 1024**3:.2f} GB to read")
    return client.query(sql).to_dataframe()


def save(df, name):
    # LF line endings on every OS, so a re-run matches the committed file byte for byte.
    df.to_csv(RAW_DIR / f"{name}.csv", index=False, encoding="utf-8", lineterminator="\n")

In [ ]:
real_sessions = run_query("real_sessions")
by_device = run_query("sessions_by_device")
by_page = run_query("sessions_by_page")

In [ ]:
# The numbers shown on the site. The sample never changes, so a re-run must match them.
real_total = real_sessions.sessions.iloc[0]
device_total = by_device.sessions.sum()
page_total = by_page.sessions.sum()

assert real_total == 333_683
assert page_total == 884_352

# Device is session-scoped, so its rows must add up exactly; page is event-scoped, so they can't.
assert device_total == real_total

summary = pd.DataFrame(
    {"sessions": [real_total, device_total, page_total]},
    index=["Real sessions", "Sum of device rows", "Sum of page rows"],
)
summary["vs_real_total"] = summary["sessions"] - real_total
summary

In [ ]:
save(real_sessions, "real_sessions")
save(by_device, "sessions_by_device")
save(by_page, "sessions_by_page")